In [1]:
# ── Setup de path ──
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "valuation":
    ROOT = ROOT.parent.parent
elif ROOT.name in ("src", "notebooks") :
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"✅ Raiz: {ROOT}")


✅ Raiz: c:\Users\jcgerardi\Documents\Pessoais\financas\potfolio-tracker


In [10]:
# ── Imports ──
import pandas as pd
from datetime import datetime, timedelta

from src.collectors.cvm_fundamentals import (
    extract_indicators,
    calcular_lpa_vpa,
    fetch_shares_outstanding,
)
from src.collectors.bcb_currency import (
    fetch_selic_atual,
    fetch_ipca_atual,
    fetch_resumo_macro,
)
from src.scripts.dividends_store import load_dividends
from src.portfolio.positions import calculate_positions, get_open_positions
from src.valuation.bazin import bazin_batch
from src.valuation.graham import graham_batch
from src.valuation.projetivo import projetivo_batch


In [13]:
# ── Carregar transações + resolver IDs ──
from src.collectors.supabase_client import get_supabase

sb = get_supabase()

# 1. Transações
print("📂 Carregando transações...")
tx_resp = sb.table("transactions").select("*").execute()
df_tx = pd.DataFrame(tx_resp.data)
print(f"   {len(df_tx)} transações")

# 2. Assets (para ticker, nome, categoria)
print("📂 Carregando assets...")
assets_resp = sb.table("assets").select("*").execute()  # ajuste o nome se for diferente
df_assets = pd.DataFrame(assets_resp.data)

# 3. Currencies (para moeda)
print("📂 Carregando currencies...")
curr_resp = sb.table("currencies").select("*").execute()  # ajuste se necessário
df_curr = pd.DataFrame(curr_resp.data)

# ── JOIN: resolver asset_id → ticker, nome, categoria ──
df_tx = df_tx.merge(
    df_assets[["id", "ticker", "name", "category"]].rename(columns={
        "id": "asset_id",
        "name": "asset_name",
        "category": "categoria",
    }),
    on="asset_id",
    how="left",
)

# ── JOIN: resolver currency_id → moeda ──
df_tx = df_tx.merge(
    df_curr[["id", "code"]].rename(columns={
        "id": "currency_id",
        "code": "moeda",
    }),
    on="currency_id",
    how="left",
)

print(f"\n✅ Colunas após JOIN: {df_tx.columns.tolist()}")
print(f"📋 Tickers encontrados: {df_tx['ticker'].unique().tolist()[:10]}...")


📂 Carregando transações...
   658 transações
📂 Carregando assets...
📂 Carregando currencies...


KeyError: "['category'] not in index"

In [14]:
print(df_assets.columns.tolist())
print(df_assets.head(2).to_dict("records"))


['id', 'ticker', 'name', 'category_id', 'currency_id', 'is_active', 'created_at']
[{'id': 83, 'ticker': 'AAVE', 'name': 'AAVE', 'category_id': 7, 'currency_id': 1, 'is_active': True, 'created_at': '2026-04-01T13:00:55.104102+00:00'}, {'id': 84, 'ticker': 'ABCB4', 'name': 'ABCB4', 'category_id': 1, 'currency_id': 1, 'is_active': True, 'created_at': '2026-04-01T13:00:55.278006+00:00'}]


In [17]:
cat_resp = sb.table("categories").select("*").execute()
df_cat = pd.DataFrame(cat_resp.data)
print(df_cat.columns.tolist())
print(df_cat.to_dict("records"))


APIError: {'message': "Could not find the table 'public.categories' in the schema cache", 'code': 'PGRST205', 'hint': "Perhaps you meant the table 'public.asset_categories'", 'details': None}

In [4]:
# ── Indicadores macro via BCB ──
print("🏦 Buscando indicadores macro do Banco Central...\n")

macro = fetch_resumo_macro()

SELIC = macro.get("selic_meta", 14.25) / 100
DOLAR = macro.get("dolar_ptax_venda")
IPCA = macro.get("ipca_acumulado_12m")

print(f"\n📌 Selic: {SELIC:.2%}")
print(f"📌 IPCA 12m: {IPCA}%")
print(f"📌 Dólar PTAX: R$ {DOLAR}")


🏦 Buscando indicadores macro do Banco Central...

🏦 Buscando indicadores macro do Banco Central...

✅ Dólar PTAX (04-01-2026): compra R$ 5.1600 | venda R$ 5.1606
✅ SGS série 432: 31 registros
✅ Selic Meta atual: 14.75% a.a.
⚠️  Erro SGS série 433: 404 Client Error: Not Found for url: https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados?formato=json&dataInicial=03%2F03%2F2026&dataFinal=02%2F04%2F2026
⚠️  Erro SGS série 13522: 404 Client Error: Not Found for url: https://api.bcb.gov.br/dados/serie/bcdata.sgs.13522/dados?formato=json&dataInicial=03%2F03%2F2026&dataFinal=02%2F04%2F2026
⚠️  Não foi possível obter o IPCA

📊 RESUMO MACRO — Banco Central do Brasil
  💵 Dólar PTAX:  R$ 5.1606
  📈 Selic Meta:  14.75% a.a.
  📉 IPCA mensal: ?%
  📉 IPCA 12m:    ?% a.a.

📌 Selic: 14.75%
📌 IPCA 12m: None%
📌 Dólar PTAX: R$ 5.1606


In [5]:
# ── Buscar cotações atuais (Yahoo — só preço) ──
import yfinance as yf

TICKERS_BR = ["PETR4.SA", "VALE3.SA", "WEGE3.SA", "BBAS3.SA"]

precos = {}
for t in TICKERS_BR:
    try:
        info = yf.Ticker(t).fast_info
        preco = round(info["lastPrice"], 2)
        precos[t.replace(".SA", "")] = preco
        print(f"✅ {t}: R$ {preco:.2f}")
    except Exception as e:
        print(f"⚠️  {t}: {e}")

print(f"\n📊 {len(precos)} cotações obtidas")


✅ PETR4.SA: R$ 48.70
✅ VALE3.SA: R$ 82.25
✅ WEGE3.SA: R$ 50.05
✅ BBAS3.SA: R$ 22.93

📊 4 cotações obtidas


In [6]:
# ── DPA últimos 12 meses via Supabase ──
print("💰 Buscando dividendos dos últimos 12 meses no Supabase...\n")

data_12m_atras = (datetime.now() - timedelta(days=365)).strftime("%Y-%m-%d")
tickers = [t.replace(".SA", "") for t in TICKERS_BR]

DPA_12M = {}

for ticker in tickers:
    # No Supabase os tickers BR estão com .SA (conforme salvo pelo fetcher)
    ticker_sa = f"{ticker}.SA"

    df_div = load_dividends(ticker=ticker_sa, start=data_12m_atras)

    if df_div.empty:
        # Tenta sem .SA (caso tenha sido salvo assim)
        df_div = load_dividends(ticker=ticker, start=data_12m_atras)

    if df_div.empty:
        print(f"⚠️  {ticker}: sem dividendos nos últimos 12m no Supabase")
        DPA_12M[ticker] = 0.0
        continue

    # Soma o valor bruto por ação dos últimos 12 meses
    dpa = df_div["gross_per_share"].astype(float).sum()
    n_pagamentos = len(df_div)

    DPA_12M[ticker] = round(dpa, 2)
    print(f"✅ {ticker}: DPA 12m = R$ {dpa:.2f} ({n_pagamentos} pagamentos)")

print(f"\n📊 DPA calculado para {len(DPA_12M)} ativos")


💰 Buscando dividendos dos últimos 12 meses no Supabase...

✅ PETR4: DPA 12m = R$ 3.28 (4 pagamentos)
⚠️  VALE3: sem dividendos nos últimos 12m no Supabase
⚠️  WEGE3: sem dividendos nos últimos 12m no Supabase
⚠️  BBAS3: sem dividendos nos últimos 12m no Supabase

📊 DPA calculado para 4 ativos


In [7]:
# ── Fundamentos CVM ──
print("📄 Buscando fundamentos na CVM...\n")

fundamentos = {}

for ticker in tickers:
    print(f"\n{'─'*50}")
    print(f"🔎 {ticker}...")

    # Nº de ações direto da CVM (FCA / VLMO)
    shares = fetch_shares_outstanding(ticker)

    if shares is None or shares == 0:
        print(f"⚠️  {ticker}: nº de ações não encontrado — pulando")
        continue

    # LPA, VPA, ROE com dados 100% CVM
    df = calcular_lpa_vpa(ticker, shares_override=shares)

    if df.empty:
        print(f"⚠️  {ticker}: sem fundamentos na CVM")
        continue

    ultimo = df.iloc[-1]
    fundamentos[ticker] = {
        "lpa": ultimo.get("lpa"),
        "vpa": ultimo.get("vpa"),
        "roe_pct": ultimo.get("roe_pct"),
        "ano": int(ultimo.get("ano", 0)),
        "shares": shares,
    }

print(f"\n✅ {len(fundamentos)} empresas com fundamentos completos")


📄 Buscando fundamentos na CVM...


──────────────────────────────────────────────────
🔎 PETR4...
⬇️  Baixando: fca_cia_aberta_2026.zip...
⬇️  Baixando: fca_cia_aberta_2025.zip...
⬇️  Baixando: vlmo_cia_aberta_2026.zip...
⬇️  Baixando: vlmo_cia_aberta_2025.zip...
⚠️  Não foi possível obter nº de ações de PETR4 via CVM
⚠️  PETR4: nº de ações não encontrado — pulando

──────────────────────────────────────────────────
🔎 VALE3...
⬇️  Baixando: fca_cia_aberta_2026.zip...
⬇️  Baixando: fca_cia_aberta_2025.zip...
⬇️  Baixando: vlmo_cia_aberta_2026.zip...
⬇️  Baixando: vlmo_cia_aberta_2025.zip...
⚠️  Não foi possível obter nº de ações de VALE3 via CVM
⚠️  VALE3: nº de ações não encontrado — pulando

──────────────────────────────────────────────────
🔎 WEGE3...
⬇️  Baixando: fca_cia_aberta_2026.zip...
⬇️  Baixando: fca_cia_aberta_2025.zip...
⬇️  Baixando: vlmo_cia_aberta_2026.zip...
⬇️  Baixando: vlmo_cia_aberta_2025.zip...
⚠️  Não foi possível obter nº de ações de WEGE3 via CVM
⚠️  WEGE3: nº d

In [8]:
# ── Montar lista de ativos para valuation ──
MEUS_ATIVOS = []

for ticker, f in fundamentos.items():
    if ticker not in precos:
        print(f"⚠️  {ticker}: sem cotação — pulando")
        continue

    dpa = DPA_12M.get(ticker, 0)
    dy = (dpa / precos[ticker] * 100) if precos[ticker] > 0 else 0

    ativo = {
        "ticker": ticker,
        "preco_atual": precos[ticker],
        "lpa": f["lpa"] or 0,
        "lpa_atual": f["lpa"] or 0,
        "vpa": f["vpa"] or 0,
        "dpa_12m": dpa,
        "num_acoes": f["shares"],
        "taxa_crescimento": 0.08,
    }
    MEUS_ATIVOS.append(ativo)

    print(
        f"✅ {ticker}: R${precos[ticker]:.2f} | "
        f"LPA=R${f['lpa']:.2f} | VPA=R${f['vpa']:.2f} | "
        f"ROE={f['roe_pct']:.1f}% | DPA=R${dpa:.2f} (DY={dy:.1f}%) | "
        f"Ações={f['shares']:,.0f}"
    )

print(f"\n📊 {len(MEUS_ATIVOS)} ativos prontos para valuation")



📊 0 ativos prontos para valuation


In [ ]:
# ── Bazin ──
print("=" * 60)
print("💰 PREÇO TETO — BAZIN (Yield mínimo 6%)")
print("=" * 60)

df_bazin = bazin_batch(MEUS_ATIVOS, yield_minimo=0.06)

for _, row in df_bazin.iterrows():
    print(f"\n{row['ticker']}:")
    print(f"  Preço atual: R$ {row['preco_atual']:.2f}")
    if row.get("preco_teto_bazin"):
        print(f"  DPA 12m: R$ {row['dpa_12m']:.2f}")
        print(f"  DY atual: {row['dy_atual_pct']:.1f}%")
        print(f"  Preço teto Bazin: R$ {row['preco_teto_bazin']:.2f}")
        print(f"  Margem: {row['margem_seguranca_pct']:+.1f}%")
    print(f"  {row['status']}")


In [ ]:
# ── Graham (Selic via BCB) ──
print("=" * 60)
print(f"📐 PREÇO JUSTO — GRAHAM (Selic: {SELIC:.2%})")
print("=" * 60)

df_graham = graham_batch(MEUS_ATIVOS, taxa_selic=SELIC)

for _, row in df_graham.iterrows():
    print(f"\n{row['ticker']}:")
    print(f"  Preço atual: R$ {row['preco_atual']:.2f}")
    if row.get("preco_graham_classico"):
        print(f"  LPA: R$ {row['lpa']:.2f} | VPA: R$ {row['vpa']:.2f}")
        print(f"  Graham clássico: R$ {row['preco_graham_classico']:.2f}")
        if row.get("preco_graham_modificado"):
            print(f"  Graham modificado: R$ {row['preco_graham_modificado']:.2f}")
        print(f"  Margem: {row['margem_seguranca_pct']:+.1f}%")
    print(f"  {row['status']}")


In [ ]:
# ── Projetivo (CVM) ──
print("=" * 60)
print("📈 PREÇO PROJETIVO — Crescimento do Lucro (CVM)")
print("=" * 60)

df_proj = projetivo_batch(
    MEUS_ATIVOS,
    anos_projecao=5,
    pl_justo=10,
    taxa_desconto=SELIC + 0.005,
)

for _, row in df_proj.iterrows():
    print(f"\n{row['ticker']}:")
    print(f"  Preço atual: R$ {row['preco_atual']:.2f}")
    if row.get("cagr_lucro_pct"):
        print(f"  CAGR do lucro (CVM): {row['cagr_lucro_pct']:.1f}% a.a.")
    if row.get("preco_projetivo"):
        print(f"  Preço projetivo: R$ {row['preco_projetivo']:.2f}")
        print(f"  Margem: {row['margem_seguranca_pct']:+.1f}%")
    print(f"  {row['status']}")


In [ ]:
# ── Resumo consolidado ──
print("\n" + "=" * 60)
print("📋 RESUMO — VALUATION COMPLETO")
print("=" * 60)
print(f"📌 Fundamentos: CVM (DFP + BPA + BPP + FCA)")
print(f"📌 Nº de ações: CVM (FCA / VLMO — automático)")
print(f"📌 Dividendos: Supabase (coletados do Yahoo Finance)")
print(f"📌 Cotações: Yahoo Finance")
print(f"📌 Selic: {SELIC:.2%} | IPCA 12m: {IPCA}% | Dólar: R$ {DOLAR}")
print("=" * 60)

for ativo in MEUS_ATIVOS:
    tk = ativo["ticker"]
    f = fundamentos[tk]

    print(f"\n{'─'*50}")
    print(f"🔹 {tk} — Preço atual: R$ {ativo['preco_atual']:.2f}")
    print(f"   LPA: R$ {ativo['lpa']:.2f} | VPA: R$ {ativo['vpa']:.2f}")
    print(f"   ROE: {f['roe_pct']:.1f}% | DPA 12m: R$ {ativo['dpa_12m']:.2f}")
    print(f"   Ações: {f['shares']:,.0f} | Ref. balanço: {f['ano']}")

    # Bazin
    bz = df_bazin[df_bazin["ticker"] == tk]
    if not bz.empty and bz.iloc[0].get("preco_teto_bazin"):
        print(
            f"   💰 Bazin:     R$ {bz.iloc[0]['preco_teto_bazin']:.2f}  "
            f"{bz.iloc[0]['status']}"
        )

    # Graham
    gr = df_graham[df_graham["ticker"] == tk]
    if not gr.empty and gr.iloc[0].get("preco_graham_classico"):
        print(
            f"   📐 Graham:    R$ {gr.iloc[0]['preco_graham_classico']:.2f}  "
            f"{gr.iloc[0]['status']}"
        )

    # Projetivo
    if not df_proj.empty:
        pr = df_proj[df_proj["ticker"] == tk]
        if not pr.empty and pr.iloc[0].get("preco_projetivo"):
            print(
                f"   📈 Projetivo: R$ {pr.iloc[0]['preco_projetivo']:.2f}  "
                f"{pr.iloc[0]['status']}"
            )

print(f"\n{'='*60}")
print(f"📅 Gerado em: {pd.Timestamp.now().strftime('%d/%m/%Y %H:%M')}")
print("🔢 ZERO dados hardcoded — tudo automático")
print(f"   🏦 Macro: BCB | 📄 Fundamentos: CVM | 💰 Dividendos: Supabase")
